# T1 prototype: sub-threshold deletion on 100 ESC-50 clips

This notebook shows the results of the job `ast-esc50-v2` (see `jobs/JOBS.md`). All logic is in `src/inaudible/`.

The pipeline:

1. Compute the masking threshold of each clip (MPEG-1 psychoacoustic model 1, 96 dB SPL for a full-scale sine).
2. Remove every STFT bin below the threshold and resynthesize the waveform with the phase kept.
3. Give the original and the changed clips to AST and compare the outputs.

The controls: `resynthesis` (STFT and inverse STFT without a change), `random_deletion` (the same number of bins per frame at random lines) and `between_clips` (the output of a different clip). The refuting result is in `docs/refuting_result.md`.

To get the outputs, run `just fetch 2026-11-06_02_ast-esc50-v2`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import Audio, Markdown, display

from inaudible.data import read_audio
from inaudible.deletion import analyze, delete_subthreshold
from inaudible.masking import Model1

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "jobs/runs/2026-11-06_02_ast-esc50-v2/output/ast-esc50-v2"
cfg = yaml.safe_load((OUT / "config.yaml").read_text())
env = json.loads((OUT / "environment.json").read_text())
metrics = json.loads((OUT / "metrics.json").read_text())
summary = pd.read_csv(OUT / "summary.csv")
clips = pd.read_csv(OUT / "clips.csv")
print("commit", env["commit"], "| device", env["device"], "| model", env["model"])
print("versions", env["versions"])
print({k: round(v, 1) for k, v in metrics.items() if k.startswith("time")}, "| clips", metrics["n_clips"])

## How much the deletion removes, and is the change inaudible?

- `fraction_bins_deleted`, `fraction_energy_deleted`: the share of STFT bins and of energy below the threshold.
- `nmr_db_*`: the mean noise-to-mask ratio of the residual (original minus changed clip) against the threshold of the original. A value below 0 dB means that the residual is under the threshold on average.
- `fraction_bins_audible_*`: the share of bins where the residual is above the threshold.
- `fraction_bands_audible_*`: the share of (frame, critical band) cells where the residual power summed over the band is above the minimum threshold in the band. MPEG layer II uses this rule for the quantization noise.

In [ ]:
def fmt(df):
    return df.assign(value=lambda d: [f"{m:.4g} [{lo:.4g}, {hi:.4g}]" for m, lo, hi in zip(d["mean"], d.ci_low, d.ci_high)])

display(fmt(summary[summary.condition == "audio"])[["measure", "value"]].set_index("measure"))

## Change of the AST output

Each cell is the mean over the clips with the 95 % bootstrap interval. `top1_changed` is the share of clips with a different top-1 class.

In [ ]:
change = fmt(summary[summary.condition != "audio"])
order = ["resynthesis", "deletion", "random_deletion", "between_clips"]
display(change.pivot(index="condition", columns="measure", values="value").loc[order])

In [ ]:
measures = [m for m in ["js_divergence", "top1_changed", "confidence_drop", "cosine_distance"] if m in set(summary.measure)]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)
for ax, measure in zip(axes, measures):
    d = summary[(summary.measure == measure)].set_index("condition").loc[order]
    y = np.arange(len(order))
    ax.errorbar(d["mean"], y, xerr=[d["mean"] - d.ci_low, d.ci_high - d["mean"]], fmt="o", ms=8,
                color="#3b6ea8", ecolor="#3b6ea8", elinewidth=2, capsize=0)
    ax.set_title(measure, fontsize=11)
    ax.set_xscale("symlog", linthresh=1e-2)
    ax.grid(axis="x", color="#e3e3e3", linewidth=0.8)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
axes[0].set_yticks(y, order)
axes[0].invert_yaxis()
fig.suptitle("Change of the AST output against the original clip (mean, 95 % bootstrap interval)", fontsize=12)
fig.tight_layout()

## The verdict: is the deletion change within the between-clip control?

The rule is in `docs/refuting_result.md`. The ratio is the mean deletion change divided by the mean between-clip change, with its 95 % bootstrap interval. The verdict is "within" if the upper limit is below the margin, and "outside" if the lower limit is above it. The main measure is the JS divergence.

In [ ]:
ratio = summary[summary.condition.str.endswith("_vs_between")]
display(Markdown(f"Margin: **{cfg['metrics']['within_margin']:.0%}** of the between-clip change."))
display(ratio.pivot(index="condition", columns="measure", values="verdict"))
display(fmt(ratio).pivot(index="condition", columns="measure", values="value"))

## Clips with a different top-1 class after the deletion

In [ ]:
cols = ["filename", "category", "top1_original", "top1_deletion", "js_divergence_deletion", "fraction_energy_deleted"]
display(clips.loc[clips.top1_changed_deletion == 1, cols].sort_values("js_divergence_deletion", ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(clips.fraction_energy_deleted, clips.js_divergence_deletion, s=40, color="#3b6ea8",
           edgecolor="white", linewidth=1.5)
ax.set_xscale("log")
ax.set_xlabel("share of energy deleted")
ax.set_ylabel("JS divergence (deletion)")
ax.set_title("Each dot is one clip", fontsize=11)
ax.grid(color="#e3e3e3", linewidth=0.8)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
fig.tight_layout()

## The threshold and the deletion on one clip

The DSP part runs locally on the CPU. It uses the original clip from the listening set.

In [ ]:
listen = clips[clips.listen == 1].reset_index(drop=True)
name = listen.filename[0]
x, sr = read_audio(OUT / "listening" / name.replace(".wav", "_original.wav"))
model = Model1(sr, cfg["masking"]["n_fft"], cfg["masking"]["playback_db"])
res = delete_subthreshold(x, model)
t = np.arange(res.spectrum_db.shape[1]) * (model.n_fft // 2) / sr
kw = dict(origin="lower", aspect="auto", extent=[t[0], t[-1], 0, sr / 2000])
db = dict(cmap="Blues", vmin=0, vmax=100)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
im = axes[0].imshow(res.spectrum_db, **db, **kw)
axes[0].set_title("spectrum (dB SPL)")
fig.colorbar(im, ax=axes[0])
im = axes[1].imshow(res.threshold_db, **db, **kw)
axes[1].set_title("masking threshold (dB SPL)")
fig.colorbar(im, ax=axes[1])
axes[2].imshow(res.audible, cmap="Greys", **kw)
axes[2].set_title(f"audible bins (black): {1 - res.fraction_bins_deleted:.1%}")
axes[0].set_ylabel("frequency (kHz)")
for ax in axes:
    ax.set_xlabel("time (s)")
fig.suptitle(f"{name} ({listen.category[0]})", fontsize=12)
fig.tight_layout()

In [ ]:
frame = int(np.argmax(res.spectrum_db.max(axis=0)))
f = model.freqs / 1000
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(f, res.spectrum_db[:, frame], color="#3b6ea8", lw=2, label="spectrum")
ax.plot(f, res.threshold_db[:, frame], color="#222222", lw=2, label="masking threshold")
ax.plot(f, model.quiet_db, color="#999999", lw=2, ls="--", label="threshold in quiet")
ax.set_ylim(-20, 110)
ax.set_xscale("log")
ax.set_xlabel("frequency (kHz)")
ax.set_ylabel("dB SPL")
ax.set_title(f"One frame at {t[frame]:.2f} s", fontsize=11)
ax.legend(frameon=False)
ax.grid(color="#e3e3e3", linewidth=0.8)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
fig.tight_layout()

## Listening test (10 clips)

For each clip: the original, the clip after the deletion, the deleted residual alone, and the random deletion control. The original and the deleted clip must sound the same. The random control must sound different. Play at a fixed, comfortable level. Write the result of each team member into a note.

In [ ]:
for row in listen.itertuples():
    stem = row.filename[:-4]
    display(Markdown(f"**{row.category}** `{row.filename}`: top-1 {row.top1_original} -> {row.top1_deletion}"))
    for kind in ["original", "deleted", "residual", "random"]:
        print(kind)
        display(Audio(str(OUT / "listening" / f"{stem}_{kind}.wav")))